# T2 — The first shape, and its half turn

**Facts used** (module T2 and T6 of the curriculum note, placed here per
X-6's dependency order "the parity material sits exactly one return deeper
than the circle").

1. The circle is $\mathbb{R}/\mathbb{Z}$: time modulo period (T2). Its
   functions are the modes $e^{2\pi i m x}$, $m \in \mathbb{Z}$.
2. With antiperiodic closure $f(x+1) = -f(x)$ the modes have
   $m \in \mathbb{Z}+\tfrac12$ (Matsubara 1955; Scherk–Schwarz 1979; X-2, X-9).
   On a ring of $N$ phases with $\theta_{i+N} = \theta_i + \pi$ the uniform
   gradients carry winding in $\mathbb{Z}+\tfrac12$ (Bulaevskii 1977 per LC-3;
   claim `klein-twisted-gradient-xor`).
3. The half-translation $J: x \mapsto x + \tfrac12$ squares to
   $J^2 = (-1)^{2m}$ on every mode; the sign depends on the $x$-parity class,
   not on the bundle (claim `half-shift-squares-by-x-parity`;
   `scripts/verify/q_j_structure_sectors.py`).
4. One traversal cannot tell periodic from antiperiodic return: $|f|^2$ is
   unchanged after one loop either way, and $f$ itself closes after two
   (X-6: the $2\pi$-vs-$4\pi$ fact).

In [ ]:
import sys, math, json, cmath, random
from fractions import Fraction
from pathlib import Path
_root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "CATALOG.md").exists())
sys.path.insert(0, str(_root / "notebooks"))
from nbkit import ROOT, show_svg, catalog, verify, mutant_must_fail, falsify
import termplot
print("repo root:", ROOT.name)

## 1–2. Windings on a twisted ring

In [ ]:
N = 8
rows = []
for twist in (0.0, math.pi):
    ws = []
    for k in range(-2, 3):
        delta = (twist + 2 * math.pi * k) / N            # uniform bond gradient closing with the twist
        total = N * delta
        ws.append(total / (2 * math.pi))
    rows.append((twist, ws))
    print(f"twist {twist:.3f}: windings of uniform gradients = {ws}")

In [ ]:
def check(twist=math.pi, claimed_lattice=0.5):
    ws = [(twist + 2 * math.pi * k) / (2 * math.pi) for k in range(-3, 4)]
    return all(abs((w - claimed_lattice) - round(w - claimed_lattice)) < 1e-12 for w in ws)

falsify(check, {"integer-windings-on-twisted-ring": lambda: {"claimed_lattice": 0.0}})

## 3. $J^2$ on the flat Klein bottle's modes (reusing the verify script's functions)

In [ ]:
import importlib.util
spec = importlib.util.spec_from_file_location("qjs", ROOT / "scripts/verify/q_j_structure_sectors.py")
qjs = importlib.util.module_from_spec(spec); spec.loader.exec_module(qjs)

table = []
for m in (0, 0.5, 1, 1.5, 2):
    for br in ("cos", "sin"):
        f = qjs.mode(m, 1, br)
        table.append((m, br, qjs.bundle(f), qjs.j2(f)))
        print(f"m = {m:<4} {br}  bundle = {table[-1][2]:<9}  J^2 = {table[-1][3]:+d}")

In [ ]:
def check(rule="x-parity"):
    if rule == "x-parity":
        return all(s == (1 if float(m).is_integer() else -1) for m, _, _, s in table)
    return all(s == (-1 if b == "twisted" else 1) for _, _, b, s in table)

falsify(check, {"bundle-decides": lambda: {"rule": "bundle"}})

## 4. One loop is blind to the sign; two loops close

In [ ]:
def traverse(m, loops, x=0.137):
    f = lambda x: cmath.exp(2j * math.pi * m * x)
    return f(x + loops) / f(x)

for m in (1, 0.5, 1.5):
    one, two = traverse(m, 1), traverse(m, 2)
    print(f"m = {m}: after one loop f -> {one.real:+.0f} f,  |f|^2 ratio {abs(one) ** 2:.0f};  after two loops f -> {two.real:+.0f} f")

In [ ]:
def check(loops_to_close=2):
    half = [0.5, 1.5, -0.5]
    intensity_blind = all(abs(abs(traverse(m, 1)) ** 2 - 1) < 1e-12 for m in half)
    closes = all(abs(traverse(m, loops_to_close) - 1) < 1e-12 for m in half)
    return intensity_blind and closes

falsify(check, {"closes-after-one-loop": lambda: {"loops_to_close": 1}})

## Falsifier: the verify script's named mutant must fail

In [ ]:
rc, out = verify("q_j_structure_sectors.py")
assert rc == 0
rc, out = verify("q_j_structure_sectors.py", mutant="bundle-decides")
mutant_must_fail("bundle-decides", rc, out)